# 相机 3D 位置计算器

从单打场 4 个角分别量到相机的激光斜距，用多点定位算出相机精确 3D 坐标。

```
       角1 (1.37, 0)────────角2 (6.86, 0)       近端底线
           │                    │
           │       球场         │
           │                    │
       角3 (1.37, 23.77)───角4 (6.86, 23.77)    远端底线
```

In [ ]:
import numpy as np
from scipy.optimize import least_squares

# ============================================================
# ★ 在这里填入你的激光测距数据（单位：米）
# ============================================================

# 单打场 4 个角的世界坐标 (x, y, z=0)
corners = np.array([
    [1.37,  0.00, 0],   # 角1: 近端左
    [6.86,  0.00, 0],   # 角2: 近端右
    [1.37, 23.77, 0],   # 角3: 远端左
    [6.86, 23.77, 0],   # 角4: 远端右
])

# --- cam66 (近端相机) ---
# 从每个角量到 cam66 镜头的斜距（米）
cam66_distances = [
    0.0,   # d1: 角1(近端左) → cam66
    0.0,   # d2: 角2(近端右) → cam66
    0.0,   # d3: 角3(远端左) → cam66
    0.0,   # d4: 角4(远端右) → cam66
]

# --- cam68 (远端相机) ---
# 从每个角量到 cam68 镜头的斜距（米）
cam68_distances = [
    0.0,   # d5: 角1(近端左) → cam68
    0.0,   # d6: 角2(近端右) → cam68
    0.0,   # d7: 角3(远端左) → cam68
    0.0,   # d8: 角4(远端右) → cam68
]

print('请填入测量数据后重新运行此 cell')
print(f'cam66 斜距: {cam66_distances}')
print(f'cam68 斜距: {cam68_distances}')

In [ ]:
# ============================================================
# 求解相机 3D 位置
# ============================================================

def solve_camera_position(corners, distances, name, initial_guess=None):
    """从 4 个已知点的斜距求解相机 3D 位置。
    
    方程: di² = (cx - xi)² + (cy - yi)² + (cz - 0)²
    4 个方程 3 个未知数 → 最小二乘
    """
    distances = np.array(distances)
    
    if np.all(distances == 0):
        print(f'⚠️ {name}: 数据全为 0，请先填入测量值')
        return None
    
    def residuals(cam_pos):
        cx, cy, cz = cam_pos
        predicted = np.sqrt(
            (corners[:, 0] - cx)**2 + 
            (corners[:, 1] - cy)**2 + 
            cz**2
        )
        return predicted - distances
    
    # 初始猜测
    if initial_guess is None:
        initial_guess = [4.1, -5.0, 6.0]  # 近端相机默认
    
    # 约束: z > 0 (相机在地面以上)
    result = least_squares(
        residuals, 
        initial_guess,
        bounds=([-np.inf, -np.inf, 0.1], [np.inf, np.inf, 20.0]),
    )
    
    cx, cy, cz = result.x
    res = result.fun  # residuals
    rmse = np.sqrt(np.mean(res**2))
    
    print(f'\n{"=" * 50}')
    print(f'{name} 相机位置:')
    print(f'  X = {cx:.3f} m  (球场宽度方向, 中线=4.115)')
    print(f'  Y = {cy:.3f} m  (球场长度方向, 底线=0/23.77)')
    print(f'  Z = {cz:.3f} m  (高度)')
    print(f'\n  RMSE = {rmse*100:.1f} cm')
    print(f'  各角残差: {["{:.1f}cm".format(r*100) for r in res]}')
    print(f'{"=" * 50}')
    
    # 验证: 用算出的位置反算斜距，和测量值对比
    print(f'\n  验证 (计算斜距 vs 测量斜距):')
    for i, (corner, d_measured) in enumerate(zip(corners, distances)):
        d_calc = np.sqrt((cx - corner[0])**2 + (cy - corner[1])**2 + cz**2)
        err = abs(d_calc - d_measured)
        print(f'    角{i+1}: 测={d_measured:.3f}m  算={d_calc:.3f}m  差={err*100:.1f}cm {"✅" if err < 0.05 else "⚠️"}')
    
    return result.x

# 求解 cam66
pos66 = solve_camera_position(
    corners, cam66_distances, 'cam66',
    initial_guess=[4.1, -5.0, 6.0]  # 近端相机
)

# 求解 cam68
pos68 = solve_camera_position(
    corners, cam68_distances, 'cam68',
    initial_guess=[4.1, 28.0, 5.5]  # 远端相机
)

In [ ]:
# ============================================================
# 3D 可视化
# ============================================================
import plotly.graph_objects as go

fig = go.Figure()

# 球场
SX_MIN, SX_MAX = 1.37, 6.86
CL = 23.77
court_x = [SX_MIN, SX_MAX, SX_MAX, SX_MIN, SX_MIN]
court_y = [0, 0, CL, CL, 0]
fig.add_trace(go.Scatter3d(
    x=court_x, y=court_y, z=[0]*5,
    mode='lines', line=dict(color='white', width=3), name='Court'
))

# 网
NET_Y = 11.885
fig.add_trace(go.Scatter3d(
    x=[SX_MIN-0.5, SX_MAX+0.5], y=[NET_Y, NET_Y], z=[0.914, 0.914],
    mode='lines', line=dict(color='gray', width=3), name='Net'
))

# 4 个角点
fig.add_trace(go.Scatter3d(
    x=corners[:, 0], y=corners[:, 1], z=corners[:, 2],
    mode='markers+text',
    marker=dict(size=6, color='yellow'),
    text=['角1(近左)', '角2(近右)', '角3(远左)', '角4(远右)'],
    textposition='top center',
    name='测量角点'
))

# 相机位置
if pos66 is not None:
    fig.add_trace(go.Scatter3d(
        x=[pos66[0]], y=[pos66[1]], z=[pos66[2]],
        mode='markers+text',
        marker=dict(size=10, color='red', symbol='diamond'),
        text=[f'cam66<br>({pos66[0]:.2f}, {pos66[1]:.2f}, {pos66[2]:.2f})'],
        textposition='top center',
        name='cam66'
    ))
    # 画从 cam66 到每个角的连线
    for i, c in enumerate(corners):
        fig.add_trace(go.Scatter3d(
            x=[pos66[0], c[0]], y=[pos66[1], c[1]], z=[pos66[2], 0],
            mode='lines', line=dict(color='red', width=1, dash='dot'),
            showlegend=False
        ))

if pos68 is not None:
    fig.add_trace(go.Scatter3d(
        x=[pos68[0]], y=[pos68[1]], z=[pos68[2]],
        mode='markers+text',
        marker=dict(size=10, color='blue', symbol='diamond'),
        text=[f'cam68<br>({pos68[0]:.2f}, {pos68[1]:.2f}, {pos68[2]:.2f})'],
        textposition='top center',
        name='cam68'
    ))
    for i, c in enumerate(corners):
        fig.add_trace(go.Scatter3d(
            x=[pos68[0], c[0]], y=[pos68[1], c[1]], z=[pos68[2], 0],
            mode='lines', line=dict(color='blue', width=1, dash='dot'),
            showlegend=False
        ))

fig.update_layout(
    title='相机位置 3D 可视化',
    scene=dict(
        xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)',
        aspectmode='data',
        camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0)),
    ),
    width=900, height=700,
)
fig.show()

In [ ]:
# ============================================================
# 生成 config.yaml 更新值
# ============================================================

if pos66 is not None and pos68 is not None:
    baseline = np.sqrt(np.sum((pos66 - pos68)**2))
    
    print('将以下值更新到 config.yaml:')
    print()
    print('cameras:')
    print('  cam66:')
    print(f'    position_3d: [{pos66[0]:.4f}, {pos66[1]:.4f}, {pos66[2]:.4f}]')
    print('  cam68:')
    print(f'    position_3d: [{pos68[0]:.4f}, {pos68[1]:.4f}, {pos68[2]:.4f}]')
    print()
    print(f'基线距离: {baseline:.2f} m')
    print(f'cam66 高度: {pos66[2]:.2f} m')
    print(f'cam68 高度: {pos68[2]:.2f} m')
    print(f'cam66 X偏移(离中线): {pos66[0] - 4.115:.3f} m')
    print(f'cam68 X偏移(离中线): {pos68[0] - 4.115:.3f} m')
    
    # 和之前的估计值对比
    print()
    print('对比之前的估计值:')
    print(f'  cam66: 之前=[4.094, -5.21, 6.2] 现在=[{pos66[0]:.3f}, {pos66[1]:.3f}, {pos66[2]:.3f}]')
    print(f'  cam68: 之前=[4.115, 28.97, 5.2] 现在=[{pos68[0]:.3f}, {pos68[1]:.3f}, {pos68[2]:.3f}]')
else:
    print('请先填入测量数据并运行求解 cell')

In [ ]:
# ============================================================
# 额外: 如果你还量了其他点（发球线角、网柱等），可以加在这里
# ============================================================

# 更多测量点（可选，提高精度）
extra_points = {
    # 'net_left':  {'world': [1.37, 11.885, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
    # 'net_right': {'world': [6.86, 11.885, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
    # 'serve_near_left':  {'world': [1.37, 5.485, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
    # 'serve_near_right': {'world': [6.86, 5.485, 0], 'cam66_dist': 0.0, 'cam68_dist': 0.0},
}

if extra_points:
    all_corners = np.vstack([corners] + [np.array(v['world']).reshape(1,3) for v in extra_points.values()])
    all_d66 = cam66_distances + [v['cam66_dist'] for v in extra_points.values()]
    all_d68 = cam68_distances + [v['cam68_dist'] for v in extra_points.values()]
    
    print('加入额外测量点后重新求解:')
    pos66_v2 = solve_camera_position(all_corners, all_d66, 'cam66 (enhanced)', initial_guess=list(pos66))
    pos68_v2 = solve_camera_position(all_corners, all_d68, 'cam68 (enhanced)', initial_guess=list(pos68))
else:
    print('取消注释 extra_points 中的行并填入数据，可以提高定位精度')